# Surrogate-Assisted CMA-ES on COCO/BBOB — Colab Runbook

Runs the full Stage-1 experiment on Colab with **Google-Drive-backed, resume-safe** output.

**Philosophy:** this VM *will* die mid-run (idle disconnect / ~12 h cap). The job is built
to survive that — mount before you write, symlink a **fresh** output dir to Drive, checkpoint
per unit, and just **rerun the cells** after a disconnect to continue from the last completed unit.

**How to use:** run the cells **top to bottom, in order**. Each cell states **Expect** (what good
output looks like) and **STOP if** (the failure signal + fix). To start a brand-new run from
scratch, change `RUN_NAME` in Cell 3.

Resume unit = (model x evolution-control x dimension x function) ~ 15 short runs. A dead VM
loses at most one unit (minutes); completed units persist on Drive and are skipped on rerun.


## Cell 1 - Runtime check (informational)

**Expect:** a GPU line *or* `No GPU (CPU runtime)`, plus CPU count and RAM.
**STOP if:** RAM < ~12 GB -> Runtime > Change runtime type > High-RAM.

> This workload is mostly **CPU-bound** (sklearn GP, small torch nets, CMA-ES, COCO). A GPU is
> *optional* - it only helps the transformer, whose weights are already committed. A High-RAM
> **CPU** runtime is a perfectly good choice.

In [ ]:
import multiprocessing, os
print("CPUs:", multiprocessing.cpu_count())
try:
    import psutil; print("RAM GB:", round(psutil.virtual_memory().total/1e9, 1))
except Exception:
    os.system("head -1 /proc/meminfo")
if os.system("nvidia-smi -L") != 0:
    print("No GPU (CPU runtime) - fine for this workload")

## Cell 2 - Mount Google Drive **FIRST**

**Expect:** `Mounted at /content/drive`.
**STOP if:** not mounted - re-run and approve the auth popup.

> Mounting **before** creating any folder under the mount point is critical: writing to a mount
> path before mounting silently lands on the ephemeral local disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 - Configure the run

**Expect:** prints the repo URL and the Drive run directory (created under it).
**STOP if:** the Drive path is not under `/content/drive/` - re-check Cell 2.

> Change `RUN_NAME` to start a **fresh** run; keep it the same to **resume**.

In [ ]:
REPO_URL   = "https://github.com/abbas-ahmad-cowlar/coco-bbob-optimizer.git"
RUN_NAME   = "stage1_noiseless_run1"   # <- change for a brand-new run
DRIVE_ROOT = f"/content/drive/MyDrive/coco_bbob_runs/{RUN_NAME}"

import os
assert DRIVE_ROOT.startswith("/content/drive/"), "Drive not mounted? Re-run Cell 2."
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Repo:     ", REPO_URL)
print("Drive run:", DRIVE_ROOT)

## Cell 4 - Clone the repo (fresh) + verify the exact code version

**Expect:** clone succeeds; prints a short commit hash and the latest commit line.
**STOP if:** clone fails (bad URL / private repo) - fix access, then re-run.

In [ ]:
%cd /content
!rm -rf coco-bbob-optimizer
!git clone {REPO_URL}
%cd /content/coco-bbob-optimizer
print("commit:")
!git rev-parse --short HEAD
!git log --oneline -1

## Cell 5 - Install pinned requirements + verify

**Expect:** finishes, then prints versions for cma / cocoex / cocopp / torch / numpy.
**STOP if:** import errors below the install (re-run once; coco-experiment builds from source,
1-2 min).

> Lower bounds are chosen so this does **not** upgrade Colab's preinstalled numpy/torch.

In [ ]:
!pip install -q -r requirements.txt
import cma, cocoex, cocopp, torch, numpy, sklearn
print("cma", cma.__version__, "| cocoex", cocoex.__version__, "| cocopp", cocopp.__version__)
print("torch", torch.__version__, "| numpy", numpy.__version__, "| sklearn", sklearn.__version__)

## Cell 6 - Verify inputs: transformer weights + COCO + a model

**Expect:** weights ~2.1 MB, `COCO ok`, and `model ok` for gp and pfn_transformer.
**STOP if:** weights missing or any error - the run cannot proceed.

> A missing input must fail loudly in 5 seconds, not 3 hours into the run.

In [ ]:
import sys, os, numpy as np
sys.path.insert(0, "src")
w = "models/models/weights/pfn_bnn_transformer.pt"
assert os.path.exists(w), "Transformer weights missing!"
print("weights bytes:", os.path.getsize(w))
import cocoex
p = next(iter(cocoex.Suite("bbob", "", "dimensions: 2 function_indices: 1 instance_indices: 1")))
print("COCO ok:", p.id)
from sacma.surrogates import make_surrogate
for m in ["gp", "pfn_transformer"]:
    s = make_surrogate(m, random_state=0)
    X = np.random.randn(20, 2); s.fit(X, (X**2).sum(1)); s.predict(X[:3])
    print("model ok:", m)

## Cell 7 - Smoke test to a **throwaway** local dir (NOT Drive)

**Expect:** ends with `SMOKE OK` and an overall table; exercises run -> analyze end-to-end.
**STOP if:** any traceback - fix before the real run.

> Output goes to ephemeral `/content` and is deleted at the end, so it never touches Drive.

In [ ]:
%cd /content/coco-bbob-optimizer
!python scripts/run_experiment.py --models gp pfn_transformer --ecs lq --functions "1,15" --dims 2 --instances "1-2" --budget-mult 30
!python scripts/analyze.py --checkpoints 15 30 --budget-mult 30
print("SMOKE OK")
!rm -rf exdata results ppdata

## Cell 8 - Symlink outputs to Drive (**fresh**) + verify arrows + marker count

**Expect:** three `->` arrows into Drive, and a completion-marker count.
- **0** = brand-new run (fresh).
- **> 0** = resuming; that many units already done and will be skipped (expected).

**STOP if:** you do **not** see `->` arrows into `/content/drive/...` (the symlink trap: a
non-empty local dir would nest the link inside it). This cell `rm -rf`s the local dirs first to
avoid that; if arrows are missing, re-run it.

In [ ]:
import os, glob, json, subprocess, datetime
for d in ["exdata", "results/diagnostics", "results/processed"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
%cd /content/coco-bbob-optimizer
!rm -rf exdata results
!mkdir -p results
!ln -sfn {DRIVE_ROOT}/exdata exdata
!ln -sfn {DRIVE_ROOT}/results/diagnostics results/diagnostics
!ln -sfn {DRIVE_ROOT}/results/processed results/processed
print("--- symlinks (must show -> into Drive) ---")
!ls -la exdata results | grep -E "exdata|diagnostics|processed"
n_done = len(glob.glob(f"{DRIVE_ROOT}/results/diagnostics/*.done"))
print("\ncompletion markers on Drive:", n_done, "(0 = fresh, >0 = resuming)")
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
json.dump({"commit": commit, "run_name": RUN_NAME, "started": datetime.datetime.now().isoformat()},
          open(f"{DRIVE_ROOT}/MANIFEST.json", "w"), indent=2)
print("manifest commit:", commit)

## Cell 9 - FULL RUN (foreground, resumable) - **rerun this cell after any disconnect**

**Expect:** progress lines `[N/2160] ... run, ... skipped`. On a rerun after a disconnect,
finished units print as skipped and it continues.
**STOP if:** the *same* unit errors repeatedly (check `run.log` on Drive).

> Foreground on purpose (never background with `&` - it would be orphaned when the cell ends).
> The cell blocks for hours; that's expected. If the VM dies, just run it again - `.done`
> markers on Drive make it resume. Units run dimension-ascending (low-D first). To scope it,
> edit `configs/stage1_noiseless.yaml` or append e.g. `--dims 2 3 5`.

In [ ]:
%cd /content/coco-bbob-optimizer
!python scripts/run_experiment.py --config configs/stage1_noiseless.yaml 2>&1 | tee -a {DRIVE_ROOT}/run.log

## Cell 10 - Analyze: tables, stats, plots, baselines, ECDF -> Drive

**Expect:** writes CSV tables + convergence PNGs to `results/processed` (Drive), fetches the four
baselines, builds the cocopp ECDF report, copies `ppdata` to Drive.
**STOP if:** a traceback - partial results are fine to analyze, but note any missing units.

> Safe to run any time, even mid-experiment, to inspect progress.

In [ ]:
%cd /content/coco-bbob-optimizer
!python scripts/analyze.py --baselines --cocopp 2>&1 | tee -a {DRIVE_ROOT}/analyze.log
!cp -r ppdata {DRIVE_ROOT}/ 2>/dev/null && echo "ppdata copied to Drive" || echo "no ppdata yet"
print("Processed outputs on Drive:", DRIVE_ROOT + "/results/processed")

## Cell 11 - Verify completeness + provenance (bring-it-home check)

**Expect:** completed-unit count vs the full Stage-1 expectation (6x3x5x24 = 2160) and the
recorded commit. 100% = done; otherwise rerun Cell 9 in another session to finish.
**STOP if:** counts look wrong or the commit is not what you expect before trusting results.

In [ ]:
import glob, json
done = glob.glob(f"{DRIVE_ROOT}/results/diagnostics/*.done")
expected = 6 * 3 * 5 * 24
print(f"completed units: {len(done)} / {expected}  ({100*len(done)/expected:.1f}%)")
try:
    print("manifest:", json.load(open(f"{DRIVE_ROOT}/MANIFEST.json")))
except Exception as e:
    print("no manifest:", e)
print("results dir:", DRIVE_ROOT + "/results/processed")